In [7]:
import sys, os
sys.path.append(os.path.abspath(".."))

import torch
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

In [3]:
pt_paths = [
    r"../data/spectral/truthfulQA_Judgelabels_and_eigs_top10.pt",
    r"../data/spectral/triviaQA_Judgelabels_and_eigs_top10.pt",
    r"../data/spectral/mmlu_Judgelabels_and_eigs_top10.pt",
    r"../data/spectral/math_Judgelabels_and_eigs_top10.pt"
]

rows = []
for path in pt_paths:
    payload = torch.load(path, map_location="cpu")
    for sample_id, item in payload["data"].items():
        rows.append({
            "id": sample_id,
            "label": item.get("label", None),
            "domain": item.get("domain", None),
        })

df_labels = pd.DataFrame(rows).drop_duplicates(subset=["id"])
df_labels["label"] = df_labels["label"].astype(str).str.lower()

df_labels.head()

,id,label,domain
0,truthfulqa_00000_t0.1_ans00,hallucination,Life Sciences
1,truthfulqa_00001_t0.1_ans00,hallucination,History & Geography
2,truthfulqa_00002_t0.1_ans00,hallucination,Physical Sciences
3,truthfulqa_00003_t0.1_ans00,hallucination,Life Sciences
4,truthfulqa_00004_t0.1_ans00,hallucination,General Knowledge


In [4]:
data = torch.load("../data/processed/df_pca.pt", map_location="cpu")

X = data["X"].numpy()
ids = data["id"]
datasets = data["dataset"]

# rebuild dataframe
pc_cols = [f"PC{i+1}" for i in range(X.shape[1])]

df_pca = pd.DataFrame(X, columns=pc_cols)
df_pca.insert(0, "id", ids)
df_pca.insert(1, "dataset", datasets)

df_pca.head()

,id,dataset,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,...,PC1384,PC1385,PC1386,PC1387,PC1388,PC1389,PC1390,PC1391,PC1392,PC1393
0,truthfulqa_00000_t0.1_ans00,truthfulqa,-30.033062,30.513218,28.804989,-24.437023,-18.623959,-9.327746,-1.688647,-11.716867,...,-0.240237,-0.495906,-0.326636,0.089442,-0.281747,0.417938,0.419732,-0.344318,-0.257110,0.243547
1,truthfulqa_00001_t0.1_ans00,truthfulqa,-41.503330,9.963426,-12.280369,23.449280,-3.379167,5.100118,16.664856,7.091613,...,-0.023484,-0.212478,-0.494584,-0.513165,-0.316231,-0.542960,0.088251,0.281707,-0.000039,0.331964
2,truthfulqa_00002_t0.1_ans00,truthfulqa,-40.586487,14.426265,11.908709,-10.016600,9.774280,-8.877830,-8.240797,-0.351241,...,-0.749912,0.491576,0.083951,0.747851,-0.421894,0.788515,-0.788881,0.415017,-0.129002,-0.164531
3,truthfulqa_00003_t0.1_ans00,truthfulqa,-44.032581,8.705839,-7.369984,-4.381597,6.520583,1.212528,-7.974981,9.117405,...,0.386629,0.342452,-0.128083,0.370429,-0.172539,-0.451733,0.808168,0.175775,-0.379351,-0.384721
4,truthfulqa_00004_t0.1_ans00,truthfulqa,-44.235367,17.476067,3.255803,-17.599913,8.564972,-6.400651,-15.197990,3.964259,...,-0.292882,-0.094813,-0.288577,-0.113414,0.057697,0.639345,0.199955,0.200291,-0.392096,-0.222924


In [5]:
df_pca_labeled = df_pca.merge(df_labels, on="id", how="left")
df_pca_labeled["label"].value_counts(dropna=False)

df_pca_labeled.head()

,id,dataset,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,...,PC1386,PC1387,PC1388,PC1389,PC1390,PC1391,PC1392,PC1393,label,domain
0,truthfulqa_00000_t0.1_ans00,truthfulqa,-30.033062,30.513218,28.804989,-24.437023,-18.623959,-9.327746,-1.688647,-11.716867,...,-0.326636,0.089442,-0.281747,0.417938,0.419732,-0.344318,-0.257110,0.243547,hallucination,Life Sciences
1,truthfulqa_00001_t0.1_ans00,truthfulqa,-41.503330,9.963426,-12.280369,23.449280,-3.379167,5.100118,16.664856,7.091613,...,-0.494584,-0.513165,-0.316231,-0.542960,0.088251,0.281707,-0.000039,0.331964,hallucination,History & Geography
2,truthfulqa_00002_t0.1_ans00,truthfulqa,-40.586487,14.426265,11.908709,-10.016600,9.774280,-8.877830,-8.240797,-0.351241,...,0.083951,0.747851,-0.421894,0.788515,-0.788881,0.415017,-0.129002,-0.164531,hallucination,Physical Sciences
3,truthfulqa_00003_t0.1_ans00,truthfulqa,-44.032581,8.705839,-7.369984,-4.381597,6.520583,1.212528,-7.974981,9.117405,...,-0.128083,0.370429,-0.172539,-0.451733,0.808168,0.175775,-0.379351,-0.384721,hallucination,Life Sciences
4,truthfulqa_00004_t0.1_ans00,truthfulqa,-44.235367,17.476067,3.255803,-17.599913,8.564972,-6.400651,-15.197990,3.964259,...,-0.288577,-0.113414,0.057697,0.639345,0.199955,0.200291,-0.392096,-0.222924,hallucination,General Knowledge


In [6]:
df_pca_labeled["label"].unique()

array(['hallucination', 'correct', 'refused', 'illogical'], dtype=object)

In [8]:
# df_pca_labeled should have columns: PC1..PCn and a label column with:
# ['hallucination', 'correct', 'refused', 'illogical']

LABEL_COL = "label"  # <-- change if your column name is different

# -------------------------
# 1) Keep only the 4 classes (drop NaN/other)
# -------------------------
classes = ["hallucination", "correct", "refused", "illogical"]

df_train = df_pca_labeled.copy()
df_train[LABEL_COL] = df_train[LABEL_COL].astype(str).str.lower()
df_train = df_train[df_train[LABEL_COL].isin(classes)].dropna(subset=[LABEL_COL])

# -------------------------
# 2) Build X (PC columns) and y (4-class labels)
# -------------------------
pc_cols = [c for c in df_train.columns if c.startswith("PC")]
X = df_train[pc_cols].to_numpy()
y = df_train[LABEL_COL].to_numpy()

print("X shape:", X.shape)
print("y distribution:\n", pd.Series(y).value_counts())

X shape: (6817, 1393)
y distribution:
 correct          2949
hallucination    2164
illogical        1587
refused           117
Name: count, dtype: int64


In [11]:
# -------------------------
# 3) Train/test split (stratified)
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=33,
    stratify=y
)

# -------------------------
# 4) Multi-class classifier
#    (Logistic regression supports multinomial)
# -------------------------
log_clf = Pipeline([
    ("scaler", StandardScaler()),  # PCs are already scaled-ish, but this is safe
    ("lr", LogisticRegression(
        max_iter=5000,
        class_weight="balanced",   # helps if classes are imbalanced
        multi_class="multinomial",
        solver="lbfgs",
        # l1_ratio=0.5,              # elastic net mixing
    ))
])

log_clf.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('lr',
                 LogisticRegression(class_weight='balanced', max_iter=5000,
                                    multi_class='multinomial'))])

In [12]:
# -------------------------
# 5) Evaluate
# -------------------------
y_pred = log_clf.predict(X_test)

print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=3))

print("\nConfusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(confusion_matrix(y_test, y_pred, labels=classes),
                   index=[f"true_{c}" for c in classes],
                   columns=[f"pred_{c}" for c in classes]))


Classification report:
               precision    recall  f1-score   support

      correct      0.567     0.537     0.552       590
hallucination      0.437     0.487     0.461       433
    illogical      0.755     0.708     0.731       318
      refused      0.208     0.217     0.213        23

     accuracy                          0.556      1364
    macro avg      0.492     0.487     0.489      1364
 weighted avg      0.564     0.556     0.559      1364


Confusion matrix (rows=true, cols=pred):
                    pred_hallucination  pred_correct  pred_refused  \
true_hallucination                 211           186            11   
true_correct                       225           317             4   
true_refused                         9             5             5   
true_illogical                      38            51             4   

                    pred_illogical  
true_hallucination              25  
true_correct                    44  
true_refused                 